# 영화 랭킹 정보 조회 및 포스터 이미지 저장하기

In [ ]:
# pip install Pillow

In [12]:
import requests
from bs4 import BeautifulSoup
import re
import os

image_dir = 'images'
os.makedirs(image_dir, exist_ok=True)  # exist_ok=True로 조건문 불필요

pattern = r'[\\/:"*?<>|]'

movie_ranking = requests.get("https://www.moviechart.co.kr/rank/realtime/index/image")

if movie_ranking.status_code != 200:
    print("페이지에 연결할 수 없습니다.")
else:
    print("영화 정보를 출력합니다.")
    soup = BeautifulSoup(movie_ranking.content, 'html.parser')

    movie_title_list = soup.select(".movieBox-list .movie-title a")
    movie_image_list = soup.select(".movieBox-list .movieBox-item img")
    print(f"수집한 영화수: {len(movie_title_list)}")

    for i, (movie_title, movie_image) in enumerate(zip(movie_title_list, movie_image_list), 1):
        title = movie_title.text.strip()
        image_src = movie_image.get('src')
        print(f"{i}위: {title} | {image_src}")

        image_response = requests.get("http://www.moviechart.co.kr" + image_src)
        image_filename = re.sub(pattern, '', title)

        # PIL 없이 바이너리로 직접 저장 (더 간단)
        with open(os.path.join(image_dir, image_filename + ".png"), 'wb') as f:
            f.write(image_response.content)
        print(f"저장 완료: {image_filename}.png")

영화 정보를 출력합니다.
수집한 영화수: 20
1위: 왕과 사는 남자 | /thumb?width=178&height=267&m_code=20242837&source=https://admin.moviechart.co.kr/assets/upload/movie/260108003526_8322.jpg
저장 완료: 왕과 사는 남자.png
2위: 휴민트 | /thumb?width=178&height=267&m_code=20241266&source=https://admin.moviechart.co.kr/assets/upload/movie/260112024651_6301.jpg
저장 완료: 휴민트.png
3위: 초속 5센티미터 | /thumb?width=178&height=267&m_code=20259583&source=https://admin.moviechart.co.kr/assets/upload/movie/260213052538_7190.jpg
저장 완료: 초속 5센티미터.png
4위: 너자 2 | /thumb?width=178&height=267&m_code=20261181&source=https://admin.moviechart.co.kr/assets/upload/movie/260202063756_4036.jpg
저장 완료: 너자 2.png
5위: 슬라이드 스트럼 뮤트 | /thumb?width=178&height=267&m_code=20261450&source=https://admin.moviechart.co.kr/assets/upload/movie/260219042530_2928.jpg
저장 완료: 슬라이드 스트럼 뮤트.png
6위: 넘버원 | /thumb?width=178&height=267&m_code=20252373&source=https://admin.moviechart.co.kr/assets/upload/movie/260123065552_5690.jpg
저장 완료: 넘버원.png
7위: 햄넷 | /thumb?width=178&height=267&m_cod

# 영화 포스터 수집 예에서 포스터 원본을 저장

In [13]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image
import re
import os
from urllib.parse import urlparse, parse_qs

BASE_URL = "https://www.moviechart.co.kr"
TARGET_URL = f"{BASE_URL}/rank/realtime/index/image"
image_dir = 'images2'
os.makedirs(image_dir, exist_ok=True)
pattern = r'[\\/:"*?<>|]'

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

response = session.get(TARGET_URL, timeout=10)

if response.status_code == 200:
    print("영화 정보를 출력합니다.")
    soup = BeautifulSoup(response.content, 'html.parser')

    movie_title_list = soup.select(".movieBox-list .movie-title a")
    movie_image_list = soup.select(".movieBox-list .movieBox-item img")
    print(f"수집한 영화 수: {len(movie_title_list)}")

    for rank, (movie_title, movie_image) in enumerate(zip(movie_title_list, movie_image_list), 1):
        try:
            url = movie_image.get('src')
            parsed_url = urlparse(url)
            query_params = parse_qs(parsed_url.query)
            image_src = query_params.get('source', [None])[0]

            if not image_src:
                print(f"[{rank}위] {movie_title.text.strip()} - 이미지 URL 없음, 건너뜁니다.")
                continue

            image_response = session.get(image_src, timeout=10)
            image_response.raise_for_status()

            # ✅ PNG 변환 집중 처리: RGB 변환 후 PNG 강제 저장
            img = Image.open(BytesIO(image_response.content)).convert("RGBA")
            img = img.convert("RGB")  # PNG 저장 시 투명도 오류 방지
            image_filename = re.sub(pattern, '', movie_title.text.strip())
            save_path = os.path.join(image_dir, f"{rank:02d}_{image_filename}.png")
            img.save(save_path, format="PNG")  # format 명시로 PNG 저장 보장
            print(f"[{rank}위] {movie_title.text.strip()} → 저장 완료: {save_path}")

        except Exception as e:
            print(f"[{rank}위] {movie_title.text.strip()} - 오류 발생: {e}")

    print(f"\n총 {rank}개 영화 처리 완료")

else:
    print("페이지에 연결할 수 없습니다.")

영화 정보를 출력합니다.
수집한 영화 수: 20
[1위] 왕과 사는 남자 → 저장 완료: images2\01_왕과 사는 남자.png
[2위] 휴민트 → 저장 완료: images2\02_휴민트.png
[3위] 초속 5센티미터 → 저장 완료: images2\03_초속 5센티미터.png
[4위] 너자 2 → 저장 완료: images2\04_너자 2.png
[5위] 슬라이드 스트럼 뮤트 → 저장 완료: images2\05_슬라이드 스트럼 뮤트.png
[6위] 넘버원 → 저장 완료: images2\06_넘버원.png
[7위] 햄넷 → 저장 완료: images2\07_햄넷.png
[8위] 부흥 → 저장 완료: images2\08_부흥.png
[9위] 매드 댄스 오피스 → 저장 완료: images2\09_매드 댄스 오피스.png
[10위] 신의악단 → 저장 완료: images2\10_신의악단.png
[11위] 렌탈 패밀리: 가족을 빌려드립니다 → 저장 완료: images2\11_렌탈 패밀리 가족을 빌려드립니다.png
[12위] 몬테크리스토 백작 → 저장 완료: images2\12_몬테크리스토 백작.png
[13위] 호퍼스 → 저장 완료: images2\13_호퍼스.png
[14위] 만약에 우리 → 저장 완료: images2\14_만약에 우리.png
[15위] 폭풍의 언덕 → 저장 완료: images2\15_폭풍의 언덕.png
[16위] 점보 → 저장 완료: images2\16_점보.png
[17위] 아기 티라노 디보: 초식이지만 괜찮아! → 저장 완료: images2\17_아기 티라노 디보 초식이지만 괜찮아!.png
[18위] 직사각형, 삼각형 → 저장 완료: images2\18_직사각형, 삼각형.png
[19위] 아바타: 불과 재 → 저장 완료: images2\19_아바타 불과 재.png
[20위] 영원 → 저장 완료: images2\20_영원.png

총 20개 영화 처리 완료
